# LENR Research Collection System - Example Usage

This notebook demonstrates how to use the LENR collection system for various tasks.

## Setup

In [ ]:
from lenr_collection import (
    LENRDatabase,
    LENRCollectionSystem,
    PDFDownloader,
    PDFProcessor,
    DeepSeekVerifier
)
import pandas as pd
import asyncio

## 1. Query Existing Database

In [ ]:
# Load database
db = LENRDatabase('lenr_papers.xlsx')

# Show statistics
stats = db.get_statistics()
print(f"Total papers: {stats['total_papers']}")
print(f"Verified: {stats['verified']}")
print(f"Average score: {stats['avg_score']:.2f}")
print(f"\nSources: {stats['sources']}")

## 2. Find High-Quality Papers

In [ ]:
# Get papers with high DeepSeek scores
high_quality = db.df[db.df['DeepSeek_Score'] > 0.8]

print(f"Found {len(high_quality)} high-quality papers\n")

# Display top 5
for idx, row in high_quality.head().iterrows():
    print(f"Title: {row['Title']}")
    print(f"Authors: {row['Authors']}")
    print(f"Score: {row['DeepSeek_Score']:.2f}")
    print(f"Summary: {row['DeepSeek_Summary'][:100]}...")
    print("-" * 80)

## 3. Search by Author

In [ ]:
author_name = "Storms"
author_papers = db.df[db.df['Authors'].str.contains(author_name, na=False, case=False)]

print(f"Found {len(author_papers)} papers by {author_name}")

# Show titles
for title in author_papers['Title'].head(10):
    print(f"  • {title}")

## 4. Analyze Papers by Source

In [ ]:
import matplotlib.pyplot as plt

# Papers by source
source_counts = db.df['Source'].value_counts()

plt.figure(figsize=(10, 6))
source_counts.plot(kind='bar')
plt.title('Papers by Source')
plt.xlabel('Source')
plt.ylabel('Number of Papers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Score Distribution

In [ ]:
# Plot score distribution
verified_papers = db.df[db.df['Verified'] == True]

plt.figure(figsize=(10, 6))
plt.hist(verified_papers['DeepSeek_Score'], bins=20, edgecolor='black')
plt.title('DeepSeek Score Distribution')
plt.xlabel('Score')
plt.ylabel('Number of Papers')
plt.axvline(verified_papers['DeepSeek_Score'].mean(), 
            color='red', linestyle='--', label='Mean')
plt.legend()
plt.show()

## 6. Download and Process a Single Paper

In [ ]:
# Download a single paper
downloader = PDFDownloader('pdfs')
processor = PDFProcessor()

# Example paper
url = "https://lenr-canr.org/acrobat/RothwellJcoldelectr.pdf"
title = "Cold Electrolysis"

# Download
pdf_path = downloader.download(url, title)

if pdf_path:
    # Process
    results = processor.process_pdf(str(pdf_path))
    
    print(f"Metadata: {results['metadata']}")
    print(f"\nAbstract: {results['abstract'][:200]}...")
    print(f"\nText length: {len(results['text'])} characters")

## 7. Run Collection (Small Test)

In [ ]:
# Run a small collection test
async def test_collection():
    system = LENRCollectionSystem()
    
    # Limit to 5 papers for testing
    system.config['max_papers_per_run'] = 5
    
    await system.run()

# Run in notebook
await test_collection()

## 8. Export Results

In [ ]:
# Export high-quality papers to CSV
high_quality = db.df[db.df['DeepSeek_Score'] > 0.8]
high_quality.to_csv('high_quality_papers.csv', index=False)

print(f"Exported {len(high_quality)} papers to high_quality_papers.csv")

## 9. Find Similar Titles

In [ ]:
# Find papers with similar titles
search_title = "Palladium Cold Fusion"
similar = db.find_similar_titles(search_title, threshold=70)

print(f"Papers similar to '{search_title}':\n")
for paper in similar[:5]:
    print(f"  Similarity: {paper['similarity']}%")
    print(f"  Title: {paper['title']}")
    print(f"  URL: {paper['url']}")
    print()